# P12 context saturation: in-context lessons vs a memory store\n10 ground-truth tasks x 5 runs per arm. A0 no memory; M top-3 retrieved lessons; C1 the task's own lessons in context (~2k tokens); C2 the same lessons inside ~32k tokens of other material; C3 inside ~100k. Lessons written by the agent from its P1/P1b runs with the evaluator's outcome. All arms on one 4-GPU server (131k window). Predictions: `quick_runs/predictions/P12.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
ARMS = ["P12_A0", "P12_M", "P12_C1", "P12_C2", "P12_C3"]
NAME = {"P12_A0": "A0 none", "P12_M": "M memory", "P12_C1": "C1 ctx 2k", "P12_C2": "C2 ctx 32k", "P12_C3": "C3 ctx 100k"}
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
CTX = json.load(open(REPO / "quick_runs/results/P12/contexts.json"))
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    for r in recs:
        r["arm"] = arm; r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["first_prompt_tokens"] = calls[0]["usage"]["prompt_tokens"] if calls else None
        r["peak_tokens"] = max((c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls), default=None)
        r["sum_prompt_tokens"] = sum(c["usage"]["prompt_tokens"] for c in calls)
        r["minutes"] = (ev[-1]["t"] - ev[0]["t"]) / 60
    return recs
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})

{'P12_A0': (50, {'final_answer': 50}), 'P12_M': (50, {'final_answer': 50}), 'P12_C1': (50, {'final_answer': 49, 'max_steps': 1}), 'P12_C2': (50, {'final_answer': 50}), 'P12_C3': (50, {'final_answer': 50})}


In [4]:
# Context actually given (tokens of the first prompt), per arm, mean over tasks
pd.DataFrame({NAME[a]: pd.Series([r["first_prompt_tokens"] for r in v]).describe()[["mean", "min", "max"]] for a, v in recs.items()}).round(0)

,A0 none,M memory,C1 ctx 2k,C2 ctx 32k,C3 ctx 100k
mean,1120.0,1527.0,2937.0,32902.0,100918.0
min,600.0,1000.0,1919.0,32367.0,100341.0
max,2767.0,3129.0,4772.0,34474.0,102574.0


In [5]:
# Silent Failures with consistent failures in P1: targeted checks passed (out of 5 runs)
rows = []
for t, checks in TARGET.items():
    for c in checks:
        row = {"task": t, "check": c}
        for a in ARMS:
            v = [r["checks"].get(c) for r in recs[a] if r["task_id"] == t]
            row[NAME[a]] = v.count(True)
        rows.append(row)
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / (5 * len(targeted))
print("share of targeted check-runs passed:", pooled.round(3).to_dict()); targeted

share of targeted check-runs passed: {'A0 none': 0.075, 'M memory': 0.125, 'C1 ctx 2k': 0.75, 'C2 ctx 32k': 0.55, 'C3 ctx 100k': 0.325}


,task,check,A0 none,M memory,C1 ctx 2k,C2 ctx 32k,C3 ctx 100k
0,sf_tf01_goal_imbalance,checks_class_distribution,0,0,5,3,1
1,sf_tf01_goal_imbalance,uses_class_weight,0,0,5,5,3
2,sf_tf01_goal_imbalance,uses_resampling,0,0,0,0,0
3,sf_tf01_goal_imbalance,uses_stratified_split,3,5,5,5,3
4,sf_tf02_player_rep_norm,handles_unit_suffix_in_height,0,0,4,5,4
5,sf_tf03_correlation_causation,column_x_included,0,0,5,2,1
6,sf_tf03_correlation_causation,applies_causal_reasoning,0,0,5,2,1
7,sf_tf05_credit_card_leakage,used_leakage_features,0,0,1,0,0


In [6]:
# Harm check: TF04 / TF06 leak-free (all checks; passes out of 5)
rows = []
for t in ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]:
    for c in sorted({c for r in recs["P12_A0"] if r["task_id"] == t for c in r["checks"]}):
        row = {"task": t, "check": c}
        for a in ARMS:
            v = [r["checks"].get(c) for r in recs[a] if r["task_id"] == t]
            row[NAME[a]] = f"{v.count(True)}/{len(v)}" + (f" ({v.count(None)} unk)" if None in v else "")
        rows.append(row)
harm = pd.DataFrame(rows); harm

,task,check,A0 none,M memory,C1 ctx 2k,C2 ctx 32k,C3 ctx 100k
0,sf_tf04_balance_torque_leakfree,has_torque_difference,4/5,5/5,5/5,0/5,1/5
1,sf_tf04_balance_torque_leakfree,has_torque_features,5/5,5/5,5/5,5/5,5/5
2,sf_tf06_metric_select_leakfree,selection_metric,3/5 (2 unk),4/5 (1 unk),2/5 (3 unk),5/5,5/5


In [7]:
# TF05 in detail: how many of the 4 leaking columns the final model kept (judge's list, most common over 3 draws)
def leak_kept(a, r):
    j = json.load(open(RUNS / a / r["task_id"] / r["run"] / "judge.json"))
    return len(Counter(tuple(sorted(d.get("leakage_features_used") or [])) for d in j["draws"]).most_common(1)[0][0])
tf05_kept = {NAME[a]: [leak_kept(a, r) for r in sorted(recs[a], key=lambda r: r["run"]) if r["task_id"] == "sf_tf05_credit_card_leakage"] for a in ARMS}
# TF04 in detail: runs whose executed code fits a model (.fit() call) vs apply a hand-written torque rule
def fits(a, r):
    ev = [json.loads(l) for l in open(RUNS / a / r["task_id"] / r["run"] / "events.jsonl")]
    return any(".fit(" in e["code"] for e in ev if e["event"] == "code_executed")
tf04_fit = {NAME[a]: sum(fits(a, r) for r in recs[a] if r["task_id"] == "sf_tf04_balance_torque_leakfree") for a in ARMS}
print("TF05 leaking columns kept per run:", tf05_kept); print("TF04 runs that fit a model (of 5):", tf04_fit)

TF05 leaking columns kept per run: {'A0 none': [4, 4, 4, 4, 4], 'M memory': [3, 3, 2, 4, 3], 'C1 ctx 2k': [1, 0, 1, 1, 1], 'C2 ctx 32k': [1, 4, 4, 3, 4], 'C3 ctx 100k': [4, 4, 4, 4, 4]}
TF04 runs that fit a model (of 5): {'A0 none': 4, 'M memory': 4, 'C1 ctx 2k': 3, 'C2 ctx 32k': 0, 'C3 ctx 100k': 0}


In [8]:
# DSBench: analysis questions correct (of 5) and answer consistency; modelling relative gap
rows = []
for t in [t for t in TASKS if t.startswith("ds")]:
    row = {"task": t}
    for a in ARMS:
        rs = [r for r in recs[a] if r["task_id"] == t]
        if t.startswith("dsa_"):
            row[NAME[a]] = f"{sum(r['checks'].get('correct') is True for r in rs)}/{len(rs)} (ac {answer_consistency(rs):.1f})"
        else:
            sc = [r["score"] for r in rs if r["score"] is not None]
            row[NAME[a]] = f"{statistics.mean(sc):.3f} +- {statistics.stdev(sc):.3f}" if len(sc) > 1 else str(sc)
    rows.append(row)
ds = pd.DataFrame(rows); ds

,task,A0 none,M memory,C1 ctx 2k,C2 ctx 32k,C3 ctx 100k
0,dsa_00000043_question1,2/5 (ac 0.2),2/5 (ac 0.2),3/5 (ac 0.3),1/5 (ac 0.3),0/5 (ac 1.0)
1,dsa_00000017_question29,3/5 (ac 0.4),4/5 (ac 0.6),2/5 (ac 0.4),5/5 (ac 1.0),5/5 (ac 1.0)
2,dsa_00000029_question1,5/5 (ac 1.0),5/5 (ac 1.0),5/5 (ac 1.0),5/5 (ac 1.0),5/5 (ac 1.0)
3,dsm_playground-series-s3e12,0.697 +- 0.026,0.724 +- 0.013,0.711 +- 0.014,0.717 +- 0.014,0.694 +- 0.040


In [9]:
# Test-set scores, all tasks (mean over runs)
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean")[[NAME[a] for a in ARMS]].round(4)

arm,A0 none,M memory,C1 ctx 2k,C2 ctx 32k,C3 ctx 100k
task,,,,,
dsa_00000017_question29,6.000000e-01,8.000000e-01,4.000000e-01,1.000000e+00,1.000000e+00
dsa_00000029_question1,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
dsa_00000043_question1,4.000000e-01,4.000000e-01,6.000000e-01,2.000000e-01,0.000000e+00
dsm_playground-series-s3e12,6.967000e-01,7.240000e-01,7.113000e-01,7.174000e-01,6.942000e-01
sf_tf01_goal_imbalance,7.603000e-01,7.929000e-01,7.765000e-01,7.816000e-01,7.903000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.664201e+06,4.650909e+06,4.650909e+06,4.656226e+06
sf_tf03_correlation_causation,2.769462e+07,2.769458e+07,4.755266e+06,1.851216e+07,2.310317e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
sf_tf05_credit_card_leakage,9.659000e-01,8.652000e-01,8.220000e-01,9.121000e-01,9.659000e-01


In [10]:
# Cost and process per arm
cost = pd.DataFrame([{"arm": NAME[a], "steps": r["steps"], "minutes": r["minutes"], "first_prompt_tokens": r["first_prompt_tokens"],
                      "sum_prompt_tokens": r["sum_prompt_tokens"], "peak_fill": (r["peak_tokens"] or 0) / 131072,
                      "src": r["task_id"].split("_")[0]} for a in ARMS for r in recs[a]])
cost.groupby("arm")[["steps", "minutes", "first_prompt_tokens", "sum_prompt_tokens", "peak_fill"]].mean().loc[[NAME[a] for a in ARMS]].round(2)

,steps,minutes,first_prompt_tokens,sum_prompt_tokens,peak_fill
arm,,,,,
A0 none,4.88,7.25,1120.0,17221.66,0.04
M memory,5.82,8.49,1527.4,22304.00,0.05
C1 ctx 2k,5.94,9.85,2936.9,30525.64,0.06
C2 ctx 32k,3.64,4.65,32902.0,124740.78,0.28
C3 ctx 100k,2.98,6.57,100917.5,305036.36,0.80


In [11]:
cost.groupby(["src", "arm"])[["steps", "minutes"]].mean().unstack(0).loc[[NAME[a] for a in ARMS]].round(1)

steps           minutes          
src           dsa  dsm   sf     dsa  dsm   sf
arm                                          
A0 none       8.7  6.8  2.7    15.8  7.6  2.9
M memory      9.8  8.2  3.4    18.1  6.3  4.0
C1 ctx 2k     7.8  9.6  4.4    20.0  7.3  5.2
C2 ctx 32k    5.6  4.6  2.5     9.5  3.0  2.5
C3 ctx 100k   4.5  4.0  2.1    13.8  4.5  3.3

In [12]:
summary = {"runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "harm": harm.to_dict("records"),
  "dsbench": ds.to_dict("records"), "tf05_leak_cols_kept": tf05_kept, "tf04_runs_fitting_model": tf04_fit,
  "score_mean": sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(5).to_dict("index"),
  "cost": cost.groupby("arm")[["steps", "minutes", "first_prompt_tokens", "sum_prompt_tokens", "peak_fill"]].mean().round(3).to_dict("index"),
  "cost_by_src": cost.groupby(["src", "arm"])[["steps", "minutes"]].mean().round(2).reset_index().to_dict("records"),
  "contexts": CTX}
json.dump(summary, open(REPO / "quick_runs/results/P12/summary.json", "w"), indent=1, default=str)
cost.to_csv(REPO / "quick_runs/results/P12/runs_cost.csv", index=False)
"done" 

'done'